# Streaming Pipeline — Spark Structured Streaming

Este notebook implementa la ingesta en tiempo real exigida por la guía del proyecto
(sección 3.1 del PDF oficial).

## Arquitectura

```
Productor batch (PySpark)
      │ inserta eventos sintéticos
      ▼
bronze.bronze_events_topic   ← tabla Delta que simula un topic Kafka
      │ readStream.format("delta")
      ▼
Spark Structured Streaming (Trigger.AvailableNow)
      │ writeStream
      ▼
bronze.bronze_bookings_stream  ← tabla Delta destino
```

## Notas técnicas

### ¿Por qué Delta como fuente en lugar de Kafka?

Las plataformas Kafka como servicio (Confluent Cloud, Aiven) requieren
registrar método de pago incluso para sus tiers gratuitos. Spark Structured
Streaming soporta **Delta Lake como fuente de stream**, lo que permite
implementar el patrón productor → topic → consumer sin servicios externos.
La tabla `bronze_events_topic` cumple el rol del topic Kafka.

Migrar a Kafka real requiere reemplazar el `readStream.format("delta")` por
`readStream.format("kafka")` con sus credenciales — el 95% del código
permanece igual.

### ¿Por qué `Trigger.AvailableNow()`?

Los clusters **Serverless** de Databricks no soportan triggers infinitos como
`processingTime`. `AvailableNow` procesa todos los datos disponibles en
micro-batches y termina, lo que es ideal para demos reproducibles y cumple
las restricciones del cluster Serverless.

### ¿Por qué Unity Catalog Volume para el checkpoint?

En workspaces modernos de Databricks el DBFS público está deshabilitado por
seguridad. Los checkpoints de Structured Streaming deben vivir en un volumen
de Unity Catalog (`/Volumes/<catalog>/<schema>/<volume>/...`). UC Volumes es
el reemplazo gobernado y permisado de DBFS.

## 1. Configuración

Se detecta dinámicamente el catálogo actual y se construye la ruta del
checkpoint apuntando a un Unity Catalog Volume que se creará en la siguiente
celda.

In [ ]:
# Catálogo actual (auto-detectado)
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]

# Número de eventos sintéticos a producir
N_EVENTOS        = 150
# Tabla que actúa como topic (productor escribe aquí)
TOPIC_TABLE      = "bronze.bronze_events_topic"
# Tabla destino del consumer
TARGET_TABLE     = "bronze.bronze_bookings_stream"
# Volume de UC para checkpoints de streaming
CHECKPOINT_VOLUME = f"/Volumes/{CATALOG_NAME}/bronze/streaming_checkpoints"
CHECKPOINT_PATH   = f"{CHECKPOINT_VOLUME}/bronze_bookings_stream"

print(f"Catálogo:           {CATALOG_NAME}")
print(f"Eventos a producir: {N_EVENTOS}")
print(f"Topic (origen):     {TOPIC_TABLE}")
print(f"Destino (sink):     {TARGET_TABLE}")
print(f"Checkpoint volume:  {CHECKPOINT_VOLUME}")

## 2. Limpieza previa

- Crea el volumen de UC para checkpoints si no existe (idempotente).
- Elimina las tablas destino y topic para que el notebook se pueda re-ejecutar.
- Borra el checkpoint anterior para evitar conflictos de offsets.

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS bronze.streaming_checkpoints")
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {TOPIC_TABLE}")
try:
    dbutils.fs.rm(CHECKPOINT_PATH, True)
except Exception as e:
    print(f"(checkpoint no existía, ok)")
print("Estado limpio.")

## 3. Capturar rangos válidos de IDs desde Silver

Los `user_id` y `property_id` generados deben caer dentro de los rangos
observados en Silver para que los joins en Gold no produzcan huérfanos.

In [ ]:
from pyspark.sql.functions import min as F_min, max as F_max

lim_u = spark.table("silver.silver_users").agg(
    F_min("user_id").alias("u_min"),
    F_max("user_id").alias("u_max")
).collect()[0]
USER_MIN, USER_MAX = lim_u["u_min"], lim_u["u_max"]

lim_p = spark.table("silver.silver_properties").agg(
    F_min("property_id").alias("p_min"),
    F_max("property_id").alias("p_max")
).collect()[0]
PROP_MIN, PROP_MAX = lim_p["p_min"], lim_p["p_max"]

print(f"user_id     range: [{USER_MIN}, {USER_MAX}]")
print(f"property_id range: [{PROP_MIN}, {PROP_MAX}]")

## 4. Productor — genera eventos sintéticos y los publica en el topic Delta

Usa `spark.range(N_EVENTOS)` para generar el lote de eventos y los escribe
en `bronze.bronze_events_topic` en modo append. En un entorno productivo,
esta tabla podría reemplazarse por un topic Kafka real sin cambiar el
consumer.

In [ ]:
eventos_batch = spark.range(N_EVENTOS).selectExpr(
    "(3000000 + id) AS booking_id",
    f"CAST({USER_MIN} + rand() * ({USER_MAX} - {USER_MIN}) AS BIGINT) AS user_id",
    f"CAST({PROP_MIN} + rand() * ({PROP_MAX} - {PROP_MIN}) AS BIGINT) AS property_id",
    "date_add(current_date(), CAST(rand() * 90 AS INT)) AS check_in",
    "date_add(current_date(), CAST(rand() * 90 AS INT) + CAST(1 + rand() * 13 AS INT)) AS check_out",
    "CAST(1 + rand() * 5 AS INT) AS guests_count",
    "ROUND(50 + rand() * 1450, 2) AS total_amount",
    "CASE WHEN rand() < 0.5 THEN 'confirmed' WHEN rand() < 0.8 THEN 'pending' ELSE 'cancelled' END AS status",
    "current_timestamp() AS created_at",
    "current_timestamp() AS updated_at"
)

(
    eventos_batch.write
                 .format("delta")
                 .mode("append")
                 .saveAsTable(TOPIC_TABLE)
)

print(f"Publicados {N_EVENTOS} eventos en el topic {TOPIC_TABLE}")
spark.sql(f"SELECT COUNT(*) AS eventos_en_topic FROM {TOPIC_TABLE}").show()

## 5. Consumidor — Spark Structured Streaming con `Trigger.AvailableNow()`

Lee el topic Delta como stream y escribe a `bronze.bronze_bookings_stream`.
`AvailableNow` procesa todos los micro-batches disponibles y termina,
garantizando exactly-once mediante `checkpointLocation` en el volumen UC.

In [ ]:
stream_df = (
    spark.readStream
         .format("delta")
         .table(TOPIC_TABLE)
)

print("Schema del stream:")
stream_df.printSchema()

query = (
    stream_df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", CHECKPOINT_PATH)
             .trigger(availableNow=True)
             .toTable(TARGET_TABLE)
)

print("Stream activo (AvailableNow). Procesando hasta vaciar el topic...")
query.awaitTermination()
print("Streaming completado.")

## 6. Validación — los eventos llegaron al destino

In [ ]:
%sql
SELECT COUNT(*) AS eventos_recibidos
FROM bronze.bronze_bookings_stream;

In [ ]:
%sql
SELECT *
FROM bronze.bronze_bookings_stream
ORDER BY created_at DESC
LIMIT 10;

## 7. Integración con Bronze batch

Los eventos del stream comparten esquema con `silver_bookings`, por lo que se
pueden unir a la carga batch original. Esta vista cierra el ciclo completo
Bronze (batch + streaming) → Silver → Gold.

In [ ]:
%sql
CREATE OR REPLACE VIEW bronze.bronze_bookings_all AS
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE)  AS check_in,
       CAST(check_out AS DATE) AS check_out,
       guests_count, total_amount, status,
       CAST(created_at AS TIMESTAMP) AS created_at,
       CAST(updated_at AS TIMESTAMP) AS updated_at,
       'batch'    AS origen
FROM bronze.bronze_bookings
UNION ALL
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE)  AS check_in,
       CAST(check_out AS DATE) AS check_out,
       guests_count, total_amount, status,
       CAST(created_at AS TIMESTAMP) AS created_at,
       CAST(updated_at AS TIMESTAMP) AS updated_at,
       'streaming' AS origen
FROM bronze.bronze_bookings_stream;

SELECT origen, COUNT(*) AS registros
FROM bronze.bronze_bookings_all
GROUP BY origen
ORDER BY origen;

## Conclusión

El notebook demuestra el patrón productor → topic → consumer streaming exigido
por la guía:

1. **Productor batch** genera eventos sintéticos coherentes con los rangos de
   IDs reales en Silver y los publica en una tabla Delta (`bronze_events_topic`)
   que actúa como topic.
2. **Spark Structured Streaming** consume el topic con `readStream.format("delta")`
   y `Trigger.AvailableNow()`.
3. Los eventos se persisten en `bronze.bronze_bookings_stream` con
   `checkpointLocation` en un Unity Catalog Volume para garantizar exactly-once.
4. Una vista `bronze_bookings_all` une la carga batch original con los eventos
   de streaming, permitiendo que Silver y Gold consuman la fuente combinada.

## Decisiones de diseño defendibles ante el docente

- **¿Por qué Delta como source y no Kafka?** Las plataformas Kafka como
  servicio (Confluent, Aiven) requieren método de pago. Delta es nativo de
  Databricks, soporta lectura streaming y replica el patrón topic →
  consumer exactamente igual. Migrar a Kafka es cambiar `format("delta")` por
  `format("kafka")` con credenciales.
- **¿Por qué `Trigger.AvailableNow()` y no `processingTime`?** Los clusters
  Serverless de Databricks no soportan triggers continuos. `AvailableNow`
  procesa los datos disponibles en micro-batches y termina, lo que también
  hace al notebook más reproducible para demos.
- **¿Por qué Unity Catalog Volume para el checkpoint?** DBFS público está
  deshabilitado en los workspaces modernos. UC Volumes es el reemplazo
  gobernado, soporta permisos a nivel de objeto y se integra nativamente con
  Structured Streaming.
- **¿Por qué `checkpointLocation`?** Garantiza exactly-once y permite reanudar
  el stream tras un fallo sin duplicar eventos.
- **¿Por qué `rand()` con rangos reales de Silver?** Para que los `user_id` y
  `property_id` generados existan en las dimensiones de Gold y los joins no
  produzcan registros huérfanos.